In [1]:
# === Bonus model: explore BPS -> bonus relationship ===
import pandas as pd
import numpy as np

BASE = r"C:\Users\veers\OneDrive\Documents\FPL Agent\fpl-copilot"
df = pd.read_parquet(BASE + r"\data\history\all_seasons_fixed.parquet")

# Keep real players, real appearances
d = df[(df["position"]!="AM") & (df["minutes"]>=1)].copy()
for c in ["bps","bonus","minutes","total_points"]:
    d[c] = pd.to_numeric(d[c], errors="coerce")

print("Bonus value distribution (per player-GW who appeared):")
print(d["bonus"].value_counts().sort_index())
print(f"\n% of appearances earning ANY bonus: {(d['bonus']>0).mean():.1%}")

print("\nBPS stats by bonus awarded (higher bonus should = higher BPS):")
print(d.groupby("bonus")["bps"].agg(["mean","min","max","count"]).round(1))

Bonus value distribution (per player-GW who appeared):
bonus
0    96544
1     4063
2     4002
3     4078
Name: count, dtype: int64

% of appearances earning ANY bonus: 11.2%

BPS stats by bonus awarded (higher bonus should = higher BPS):
       mean  min  max  count
bonus                       
0      10.1  -25   47  96544
1      27.9   14   64   4063
2      32.0   16   80   4002
3      40.2   20  128   4078


In [2]:
# === Learn the empirical BPS -> expected bonus curve ===
# For each BPS level, what's the AVERAGE bonus earned? (expected value, handles the relative nature)

# Bin BPS and compute mean bonus per bin
d["bps_bin"] = (d["bps"] // 5) * 5   # 5-point BPS buckets
curve = d.groupby("bps_bin").agg(
    exp_bonus=("bonus","mean"),
    n=("bonus","size"),
    p_any=("bonus", lambda x: (x>0).mean())
).reset_index()
curve = curve[curve["n"]>=30]   # stable buckets only

print("BPS -> expected bonus (empirical curve):\n")
print(f"{'BPS':>6s} {'E[bonus]':>9s} {'P(any)':>8s} {'n':>7s}")
for _, r in curve.iterrows():
    if r["bps_bin"] % 10 == 0 or r["exp_bonus"] > 0.1:  # show meaningful rows
        print(f"{int(r['bps_bin']):>6d} {r['exp_bonus']:>9.3f} {r['p_any']:>8.3f} {int(r['n']):>7d}")

# Key transition zone
print("\nThe critical zone (where bonus starts mattering):")
print(curve[(curve["bps_bin"]>=20)&(curve["bps_bin"]<=45)][["bps_bin","exp_bonus","p_any","n"]].to_string(index=False))

BPS -> expected bonus (empirical curve):

   BPS  E[bonus]   P(any)       n
   -10     0.000    0.000     501
     0     0.000    0.000   25145
    10     0.000    0.000   20699
    20     0.160    0.120    9001
    25     0.752    0.465    7988
    30     1.692    0.826    4053
    35     2.246    0.953    1708
    40     2.452    0.971     882
    45     2.668    0.994     506
    50     2.774    1.000     363
    55     2.879    1.000     207
    60     2.898    1.000     108
    65     2.959    1.000      74
    70     2.971    1.000      34

The critical zone (where bonus starts mattering):
 bps_bin  exp_bonus    p_any    n
      20   0.160093 0.119876 9001
      25   0.751627 0.464822 7988
      30   1.692327 0.826055 4053
      35   2.245902 0.953162 1708
      40   2.452381 0.970522  882
      45   2.667984 0.994071  506


In [3]:
# === What drives BPS? Correlate with the components we already predict ===
# These are the things our other models produce (or will): goals, assists, CS, minutes, defensive
drivers = ["goals_scored","assists","clean_sheets","minutes","bps"]
for c in drivers:
    d[c] = pd.to_numeric(d[c], errors="coerce")

# defensive contribution column (2025-26) if present
if "defensive_contribution" in d.columns:
    d["defensive_contribution"] = pd.to_numeric(d["defensive_contribution"], errors="coerce")

print("Correlation of each component with BPS:\n")
for c in ["goals_scored","assists","clean_sheets","minutes"]:
    print(f"  {c:16s}: {d[c].corr(d['bps']):.3f}")

# How much BPS does each action add on average? (mean BPS by count)
print("\nMean BPS by goals scored:")
print(d.groupby("goals_scored")["bps"].agg(["mean","count"]).head(4).round(1))

print("\nMean BPS by minutes bucket (playing time alone drives BPS via passes/actions):")
d["min_bucket"] = pd.cut(d["minutes"], [0,30,60,89,90], labels=["<30","30-60","60-89","90"])
print(d.groupby("min_bucket", observed=True)["bps"].mean().round(1))

Correlation of each component with BPS:

  goals_scored    : 0.601
  assists         : 0.364
  clean_sheets    : 0.387
  minutes         : 0.465

Mean BPS by goals scored:
              mean  count
goals_scored             
0             10.8  99355
1             30.0   8384
2             52.6    836
3             76.3    101

Mean BPS by minutes bucket (playing time alone drives BPS via passes/actions):
min_bucket
<30       4.2
30-60     5.8
60-89    13.6
90       16.5
Name: bps, dtype: float64


In [4]:
# === Piece 1: predict BPS from components (train on ACTUALS, ready for predicted inputs) ===
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

# Components that will exist as PREDICTIONS at assembly time
comp = ["goals_scored","assists","clean_sheets","minutes"]
# add defensive_contribution where available (2025-26) — but train on the broad set first
model_df = d.dropna(subset=comp+["bps"]).copy()

# position matters (a defender's goal = more BPS than a forward's), so add position dummies
model_df["is_def"] = (model_df["position"]=="DEF").astype(int)
model_df["is_mid"] = (model_df["position"]=="MID").astype(int)
model_df["is_gk"]  = (model_df["position"]=="GK").astype(int)
feats = comp + ["is_def","is_mid","is_gk"]

# Walk-forward: train on older seasons, test on 2024-25
seasons = sorted(model_df["season"].unique())
tr = model_df[model_df["season"] <= "2023-24"]
te = model_df[model_df["season"] == "2024-25"]

lr = LinearRegression().fit(tr[feats], tr["bps"])
pred = lr.predict(te[feats])

print("BPS prediction (from actual components):")
print(f"  MAE: {mean_absolute_error(te['bps'], pred):.2f} BPS")
print(f"  R² : {r2_score(te['bps'], pred):.3f}")
print(f"  (BPS ranges roughly -25 to 128; mean {te['bps'].mean():.1f})\n")

print("Learned weights (BPS added per unit):")
for f, w in zip(feats, lr.coef_):
    print(f"  {f:16s}: {w:+.2f}")
print(f"  intercept: {lr.intercept_:.2f}")

BPS prediction (from actual components):
  MAE: 5.39 BPS
  R² : 0.620
  (BPS ranges roughly -25 to 128; mean 11.2)

Learned weights (BPS added per unit):
  goals_scored    : +19.08
  assists         : +10.91
  clean_sheets    : +6.14
  minutes         : +0.09
  is_def          : +7.38
  is_mid          : +2.84
  is_gk           : +10.50
  intercept: -1.75


In [5]:
# === Check which extra BPS-driving columns exist ===
candidates = ["clearances_blocks_interceptions","recoveries","tackles","key_passes",
              "saves","penalties_saved","penalties_missed","own_goals","yellow_cards",
              "red_cards","goals_conceded","bonus","big_chances_created",
              "errors_leading_to_goal","defensive_contribution"]
present = [c for c in candidates if c in d.columns]
print("Available extra BPS drivers:", present)

# coverage by season (many are era-specific)
for c in present:
    d[c] = pd.to_numeric(d[c], errors="coerce")
print("\nNon-null coverage (recent seasons):")
cov = d[d["season"].isin(["2022-23","2023-24","2024-25"])].groupby("season")[present].apply(lambda g: g.notna().mean().round(2))
print(cov.T.to_string())

Available extra BPS drivers: ['clearances_blocks_interceptions', 'recoveries', 'tackles', 'key_passes', 'saves', 'penalties_saved', 'penalties_missed', 'own_goals', 'yellow_cards', 'red_cards', 'goals_conceded', 'bonus', 'big_chances_created', 'errors_leading_to_goal', 'defensive_contribution']

Non-null coverage (recent seasons):
season                           2022-23  2023-24  2024-25
clearances_blocks_interceptions      0.0      0.0      0.0
recoveries                           0.0      0.0      0.0
tackles                              0.0      0.0      0.0
key_passes                           0.0      0.0      0.0
saves                                1.0      1.0      1.0
penalties_saved                      1.0      1.0      1.0
penalties_missed                     1.0      1.0      1.0
own_goals                            1.0      1.0      1.0
yellow_cards                         1.0      1.0      1.0
red_cards                            1.0      1.0      1.0
goals_conceded    

In [6]:
# === Add available extra features + compare algorithms for BPS prediction ===
import lightgbm as lgb, xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler

extra = ["saves","yellow_cards","red_cards","goals_conceded","penalties_missed","own_goals"]
feats2 = comp + ["is_def","is_mid","is_gk"] + extra

mdf = d.dropna(subset=comp+extra+["bps"]).copy()
mdf["is_def"]=(mdf["position"]=="DEF").astype(int)
mdf["is_mid"]=(mdf["position"]=="MID").astype(int)
mdf["is_gk"]=(mdf["position"]=="GK").astype(int)

tr = mdf[mdf["season"]<="2023-24"]; te = mdf[mdf["season"]=="2024-25"]
Xtr,ytr = tr[feats2], tr["bps"]; Xte,yte = te[feats2], te["bps"]
sc = StandardScaler().fit(Xtr)

models = {
    "Linear": LinearRegression(),
    "RandomForest": RandomForestRegressor(n_estimators=200, min_samples_leaf=20, n_jobs=-1, random_state=42),
    "XGBoost": xgb.XGBRegressor(n_estimators=300, max_depth=5, learning_rate=0.05, subsample=0.8, random_state=42),
    "LightGBM": lgb.LGBMRegressor(n_estimators=300, num_leaves=31, learning_rate=0.05, random_state=42, verbose=-1),
}
print(f"{'model':14s} {'MAE':>7s} {'R2':>7s}")
print("-"*30)
# baseline: the old feature set with linear
old = LinearRegression().fit(tr[comp+['is_def','is_mid','is_gk']], ytr)
print(f"{'(old feats)':14s} {mean_absolute_error(yte, old.predict(te[comp+['is_def','is_mid','is_gk']])):>7.2f} {r2_score(yte, old.predict(te[comp+['is_def','is_mid','is_gk']])):>7.3f}")
for name,m in models.items():
    if name=="SVR":
        m.fit(sc.transform(Xtr),ytr); p=m.predict(sc.transform(Xte))
    else:
        m.fit(Xtr,ytr); p=m.predict(Xte)
    print(f"{name:14s} {mean_absolute_error(yte,p):>7.2f} {r2_score(yte,p):>7.3f}")

model              MAE      R2
------------------------------
(old feats)       5.39   0.620
Linear            5.22   0.652
RandomForest      4.23   0.742
XGBoost           4.20   0.747
LightGBM          4.19   0.747
